In [1]:
%pip install -q syncode==0.4.16 transformers==4.53.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.1/206.1 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0

In [2]:
import syncode
import transformers

print("SynCode imported successfully")
print("Transformers:", transformers.__version__)

SynCode imported successfully
Transformers: 4.53.2


In [3]:
from syncode import Syncode

grammar = r"""
start: number "," number "," operator
number: /-?[0-9]+(\.[0-9]*)?/
operator: "+" | "-" | "*" | "/"
"""

syn_llm = Syncode(
    model = "microsoft/phi-2",
    grammar = grammar,
    parse_output_only = True,
    max_new_tokens = 40
)

[2026-10-24 20:56:47,377-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-24 20:58:39,969-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_3913182923_50257.pkl exist: False
[2026-10-24 20:58:39,970-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_3913182923_50257.pkl.
[2026-10-24 20:58:40,797-syncode.mask_store.fsm_set] - 6 FSMs with 16 states initialized in 0.00 seconds
[2026-10-24 20:58:40,798-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-24 20:58:40,805-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 36 to 6


100%|██████████| 16/16 [00:05<00:00,  2.94it/s]


[2026-10-24 20:58:46,299-syncode.mask_store.mask_store] - Time taken to create mask store: 6.32 seconds


In [1]:
prompt = """Extract the two numbers and arithmetic operator.
Output only: number,number,operator

Question: What is 45.1 plus 23.54?
Answer: 45.1,23.54,+

Question: What is 120.4 divided by 4.0?
Answer: 120.4,4.0,/

Question: What is 18.5 minus 7.2?
Answer: 18.5,7.2,-

Question: What is -6.0 multiplied by 3.5?
Answer: -6.0,3.5,*"""

In [20]:
def run_syncode(textual_query):
    query_prompt = prompt + f"\nQuestion: {textual_query}\nAnswer:"
    result = syn_llm.infer(query_prompt)[0]
    a, b, operator = result.strip().split(",")

    operands = [float(a), float(b)]
    if operator == "+":
        answer = operands[0] + operands[1]
    elif operator == "-":
        answer = operands[0] - operands[1]
    elif operator == "*":
        answer = operands[0] * operands[1]
    elif operator == "/":
        answer = operands[0] / operands[1]

    return operands, operator, float(answer)

In [9]:
tests = [
    ("What is 327. multiplied by 11.0?", 3597.0),
    ("What is 45.1 plus 23.54?", 68.64),
    ("What is 120.4 divided by 4.0?", 30.1),
    ("What is 18.5 minus 7.2?", 11.3)
]

for query, expected in tests:
    operands, operator, answer = run_syncode(query)
    print(operands, operator, answer)
    assert abs(answer - expected) < 1e-5

[327.0, 11.0] * 3597.0
[45.1, 23.54] + 68.64
[120.4, 4.0] / 30.1
[18.5, 7.2] - 11.3


In [15]:
tests = [
    ("Calculate 12.5 times 8.0", 100.0),
    ("What is 99.9 minus 25.4?", 74.5),
    ("Divide 144.0 by 12.0", 12.0),
    ("Add 3.75 and 6.25", 10.0),
    ("What is -5.5 multiplied by 2.0?", -11.0)
]

for query, expected in tests:
    try:
        operands, operator, answer = run_syncode(query)
        print(query, "->", operands, operator, answer, abs(answer - expected) < 1e-5)
    except Exception as e:
        print(query, "-> ERROR:", e)

Calculate 12.5 times 8.0 -> [12.5, 8.0] * 100.0 True
What is 99.9 minus 25.4? -> [99.9, 25.4] - 74.5 True
Divide 144.0 by 12.0 -> [144.0, 12.0] / 12.0 True
Add 3.75 and 6.25 -> [3.75, 6.25] + 10.0 True
What is -5.5 multiplied by 2.0? -> [-5.5, 2.0] * -11.0 True


In [21]:
tests = [
    ("What is 327. multiplied by 11.0?", 3597.0),
    ("What is -8.5 multiplied by 4.0?", -34.0),
    ("Calculate -12.0 times 2.5", -30.0),
    ("What is 7.5 multiplied by -3.0?", -22.5)
]

for query, expected in tests:
    operands, operator, answer = run_syncode(query)
    print(operands, operator, answer, abs(answer - expected) < 1e-5)

[327.0, 11.0] * 3597.0 True
[-8.5, 4.0] * -34.0 True
[-12.0, 2.5] * -30.0 True
[7.5, -3.0] * -22.5 True
